# Zarr Store Inspector

Point this notebook at any local or GCS Zarr store to get a complete picture of its contents before writing any retrieval code.

In [1]:
from pathlib import Path
import sys

import gcsfs
import numpy as np
import xarray as xr
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("Project root:", PROJECT_ROOT)

Project root: /Users/chxmr/code/glide


## Configure Store Path

Set `STORE_PATH` to a local directory or a `gs://` URI.  
Set `USE_ANONYMOUS` to `False` if the bucket requires credentials.

In [2]:
STORE_PATH = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
USE_ANONYMOUS = True   # set False to use application-default credentials

def open_store(path: str, anonymous: bool = True) -> xr.Dataset:
    if path.startswith("gs://"):
        token = "anon" if anonymous else "google_default"
        fs = gcsfs.GCSFileSystem(token=token)
        mapper = fs.get_mapper(path)
        return xr.open_zarr(mapper, consolidated=True, chunks={})
    return xr.open_zarr(path, consolidated=True, chunks={})

ds = open_store(STORE_PATH, anonymous=USE_ANONYMOUS)
print("Store opened:", STORE_PATH)

I0402 10:25:03.884965 8804859 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0402 10:25:03.888767 8804880 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(100, generation: 1)


Store opened: gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3


## Dimensions

In [3]:
dim_rows = [{"dimension": k, "size": v} for k, v in ds.sizes.items()]
pd.DataFrame(dim_rows).set_index("dimension")

,size
dimension,
time,1323648
latitude,721
longitude,1440
level,37


## Coordinates

Shape, dtype, units/calendar where present, and first/last values.

In [4]:
coord_rows = []
for name, coord in ds.coords.items():
    attrs = coord.attrs
    vals = coord.values
    first = vals.flat[0] if vals.size > 0 else None
    last  = vals.flat[-1] if vals.size > 0 else None
    coord_rows.append({
        "coordinate": name,
        "dims": str(coord.dims),
        "shape": str(coord.shape),
        "dtype": str(coord.dtype),
        "units / calendar": attrs.get("units", attrs.get("calendar", "")),
        "first": first,
        "last": last,
    })
pd.DataFrame(coord_rows).set_index("coordinate")

,dims,shape,dtype,units / calendar,first,last
coordinate,,,,,,
latitude,"('latitude',)","(721,)",float32,degrees_north,90.0,-90.0
level,"('level',)","(37,)",int64,Hectopascal(hPa),1,1000
longitude,"('longitude',)","(1440,)",float32,degrees_east,0.0,359.75
time,"('time',)","(1323648,)",datetime64[ns],,1900-01-01T00:00:00.000000000,2050-12-31T23:00:00.000000000


## Data Variables

Shape, dtype, units, long_name, and estimated uncompressed size per variable.

In [5]:
var_rows = []
for name, da in ds.data_vars.items():
    attrs = da.attrs
    nbytes = int(np.prod(da.shape)) * np.dtype(da.dtype).itemsize
    var_rows.append({
        "variable": name,
        "dims": str(da.dims),
        "shape": str(da.shape),
        "dtype": str(da.dtype),
        "units": attrs.get("units", ""),
        "long_name": attrs.get("long_name", ""),
        "uncompressed MiB": round(nbytes / 1024**2, 1),
    })
pd.DataFrame(var_rows).set_index("variable")

,dims,shape,dtype,units,long_name,uncompressed MiB
variable,,,,,,
100m_u_component_of_wind,"('time', 'latitude', 'longitude')","(1323648, 721, 1440)",float32,m s**-1,100 metre U wind component,5242402.3
100m_v_component_of_wind,"('time', 'latitude', 'longitude')","(1323648, 721, 1440)",float32,m s**-1,100 metre V wind component,5242402.3
10m_u_component_of_neutral_wind,"('time', 'latitude', 'longitude')","(1323648, 721, 1440)",float32,m s**-1,Neutral wind at 10 m u-component,5242402.3
10m_u_component_of_wind,"('time', 'latitude', 'longitude')","(1323648, 721, 1440)",float32,m s**-1,10 metre U wind component,5242402.3
10m_v_component_of_neutral_wind,"('time', 'latitude', 'longitude')","(1323648, 721, 1440)",float32,m s**-1,Neutral wind at 10 m v-component,5242402.3
...,...,...,...,...,...,...
wave_spectral_directional_width_for_wind_waves,"('time', 'latitude', 'longitude')","(1323648, 721, 1440)",float32,radians,Wave spectral directional width for wind waves,5242402.3
wave_spectral_kurtosis,"('time', 'latitude', 'longitude')","(1323648, 721, 1440)",float32,dimensionless,Wave spectral kurtosis,5242402.3
wave_spectral_peakedness,"('time', 'latitude', 'longitude')","(1323648, 721, 1440)",float32,dimensionless,Wave spectral peakedness,5242402.3


## Chunk Layout

Understanding chunks is critical for efficient bounding-box retrieval — ideally your spatial slice aligns with chunk boundaries.

In [6]:
chunk_rows = []
for name in ds.data_vars:
    encoding = ds[name].encoding
    chunk_rows.append({
        "variable": name,
        "chunks": encoding.get("chunks", "not set"),
        "compressor": str(encoding.get("compressor", "none")),
        "dtype (stored)": encoding.get("dtype", ds[name].dtype),
    })
pd.DataFrame(chunk_rows).set_index("variable")

,chunks,compressor,dtype (stored)
variable,,,
100m_u_component_of_wind,"(1, 721, 1440)",none,float32
100m_v_component_of_wind,"(1, 721, 1440)",none,float32
10m_u_component_of_neutral_wind,"(1, 721, 1440)",none,float32
10m_u_component_of_wind,"(1, 721, 1440)",none,float32
10m_v_component_of_neutral_wind,"(1, 721, 1440)",none,float32
...,...,...,...
wave_spectral_directional_width_for_wind_waves,"(1, 721, 1440)",none,float32
wave_spectral_kurtosis,"(1, 721, 1440)",none,float32
wave_spectral_peakedness,"(1, 721, 1440)",none,float32


## Dataset-level Attributes

In [7]:
if ds.attrs:
    for k, v in ds.attrs.items():
        print(f"{k}: {v}")
else:
    print("No dataset-level attributes.")

last_updated: 2026-04-02 02:31:50.473162+00:00
valid_time_start: 1940-01-01
valid_time_stop: 2025-12-31
valid_time_stop_era5t: 2026-03-27


## Time Coverage

In [8]:
time_candidates = [c for c in ds.coords if "time" in c.lower()]
for tc in time_candidates:
    t = ds[tc]
    vals = t.values
    n = t.size
    step = (vals[1] - vals[0]) if n > 1 else "N/A"
    print(f"{tc}: {vals[0]}  →  {vals[-1]}  ({n} steps, step={step})")

time: 1900-01-01T00:00:00.000000000  →  2050-12-31T23:00:00.000000000  (1323648 steps, step=3600000000000 nanoseconds)


## Vertical Coordinate

In [9]:
level_candidates = [c for c in ds.coords if any(k in c.lower() for k in ["level", "pressure", "isobaric", "height"])]
for lc in level_candidates:
    lv = ds[lc].values
    print(f"{lc}: {lv}")

level: [   1    2    3    5    7   10   20   30   50   70  100  125  150  175
  200  225  250  300  350  400  450  500  550  600  650  700  750  775
  800  825  850  875  900  925  950  975 1000]


## Spatial Extent &amp; Resolution

In [10]:
spatial_keywords = {"latitude", "longitude", "lat", "lon"}
for coord_name in ds.coords:
    if coord_name.lower() in spatial_keywords or any(coord_name.lower().startswith(k) for k in spatial_keywords):
        vals = ds[coord_name].values
        step = round(float(np.diff(vals[:2])[0]), 6) if len(vals) > 1 else "N/A"
        print(f"{coord_name}: min={vals.min():.4f}  max={vals.max():.4f}  step={step}  n={len(vals)}")

latitude: min=-90.0000  max=90.0000  step=-0.25  n=721
longitude: min=0.0000  max=359.7500  step=0.25  n=1440


In [11]:
var_rows


[{'variable': '100m_u_component_of_wind',
  'dims': "('time', 'latitude', 'longitude')",
  'shape': '(1323648, 721, 1440)',
  'dtype': 'float32',
  'units': 'm s**-1',
  'long_name': '100 metre U wind component',
  'uncompressed MiB': 5242402.3},
 {'variable': '100m_v_component_of_wind',
  'dims': "('time', 'latitude', 'longitude')",
  'shape': '(1323648, 721, 1440)',
  'dtype': 'float32',
  'units': 'm s**-1',
  'long_name': '100 metre V wind component',
  'uncompressed MiB': 5242402.3},
 {'variable': '10m_u_component_of_neutral_wind',
  'dims': "('time', 'latitude', 'longitude')",
  'shape': '(1323648, 721, 1440)',
  'dtype': 'float32',
  'units': 'm s**-1',
  'long_name': 'Neutral wind at 10 m u-component',
  'uncompressed MiB': 5242402.3},
 {'variable': '10m_u_component_of_wind',
  'dims': "('time', 'latitude', 'longitude')",
  'shape': '(1323648, 721, 1440)',
  'dtype': 'float32',
  'units': 'm s**-1',
  'long_name': '10 metre U wind component',
  'uncompressed MiB': 5242402.3},
 

In [12]:
ds.close()